## ***1. Installing Rust***

In [1]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-07-16 for version 1.97.1 (8bab26f4f 2026-07-14)
info: downloading 6 components
        cargo downloading [               ]         0 B (0 B/s, ETA: 0s)A: 0s)  
        cargo downloading [               ]   10.63 MiB (0 B/s, ETA: 0s)
        cargo downloading [#              ]   10.63 MiB (17.21 MiB/s, ETA: 1s)
        cargo downloading [#              ]   10.63 MiB (17.21 MiB/s, ETA: 1s)
        cargo downloading [#              ]   10.63 MiB (17.21 MiB/s, ETA: 1s)
        cargo downloading [#              ]   10.63 MiB (17.21 MiB/s, ETA: 1s)

## ***2. Adding Cargo to PATH + checking***

In [2]:
import os
os.environ["PATH"] += f":{os.environ['HOME']}/.cargo/bin"

In [3]:
!rustc --version
!cargo --version

rustc 1.97.1 (8bab26f4f 2026-07-14)
cargo 1.97.1 (c980f4866 2026-06-30)


## ***3. Creating a project with smartcore***

In [4]:
import os
os.environ["PATH"] += f":{os.environ.get('HOME', '/root')}/.cargo/bin"

In [5]:
!mkdir -p /kaggle/working/rust_project/src

Cargo.toml

In [6]:
with open("/kaggle/working/rust_project/Cargo.toml", "w") as f:
    f.write('''
[package]
name = "rust_project"
version = "0.1.0"
edition = "2021"

[dependencies]
smartcore = { version = "0.5.5", features = ["datasets"] }
''')

src/main.rs

In [7]:
with open("/kaggle/working/rust_project/src/main.rs", "w") as f:
    f.write(r'''
use smartcore::dataset::iris;

fn main() {
    let iris_data = iris::load_dataset();

    println!("Dataset Iris loaded successfully!");
    println!("Number of samples : {}", iris_data.data.len() / iris_data.num_features);
    println!("Number of features: {}", iris_data.num_features);
    println!("Target names      : {:?}", iris_data.target_names);
}
''')

print("Project created.")

Project created.


## ***4. Running the code***!cd /kaggle/working/rust_project && cargo run

In [8]:
!cd /kaggle/working/rust_project && cargo run

    Updating crates.io index
     Locking 33 packages to latest compatible versions
  Downloaded approx v0.5.1                                                 
  Downloaded cfg-if v1.0.4                                                 
  Downloaded num-rational v0.4.2ytes: 23.7KiB                              
  Downloaded chacha20 v0.10.1                                              
  Downloaded num-complex v0.4.6                                            
  Downloaded getrandom v0.4.3                                              
  Downloaded erased-serde v0.4.10                                          
  Downloaded typetag v0.2.23ng bytes: 16.2KiB                              
  Downloaded unicode-ident v1.0.24es: 6.3KiB                               
  Downloaded num-iter v0.1.46g bytes: 779.8KiB                             
  Downloaded rand_core v0.10.1 bytes: 406.3KiB                             
  Downloaded serde_derive v1.0.229es: 278.4KiB                             
  Do

## ***5. Helper for more convenient relaunch (optional)***

In [9]:
import os
import textwrap
import subprocess

In [10]:
os.environ["PATH"] += f":{os.environ.get('HOME', '/root')}/.cargo/bin"

In [11]:
def run_rust(code: str, deps: str = ""):
    """Runs Rust code through Cargo."""
    project = "/kaggle/working/rust_tmp"
    os.makedirs(f"{project}/src", exist_ok=True)

    cargo_toml = f'''
[package]
name = "rust_tmp"
version = "0.1.0"
edition = "2021"

[dependencies]
{deps}
'''
    with open(f"{project}/Cargo.toml", "w") as f:
        f.write(cargo_toml)

    with open(f"{project}/src/main.rs", "w") as f:
        f.write(textwrap.dedent(code))

    result = subprocess.run(
        ["cargo", "run", "--quiet"],
        cwd=project,
        capture_output=True,
        text=True
    )

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    
    if result.returncode != 0:
        print(f"Chyba (exit code {result.returncode})")

In [12]:
run_rust('''
use smartcore::dataset::iris;

fn main() {
    let data = iris::load_dataset();
    println!("Samples: {}", data.data.len() / data.num_features);
    println!("Features: {}", data.num_features);
}
''', deps='smartcore = { version = "0.5.5", features = ["datasets"] }')

Samples: 150
Features: 4



In [14]:
run_rust('''
use smartcore::dataset::iris;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::model_selection::train_test_split;
use smartcore::ensemble::random_forest_classifier::RandomForestClassifier;
use smartcore::metrics::accuracy;

fn main() {
    let iris = iris::load_dataset();

    // DenseMatrix::new vrací Result → musíme unwrapnout
    let x = DenseMatrix::new(
        iris.num_samples,
        iris.num_features,
        iris.data.clone(),
        false, // row-major
    ).unwrap();   // <-- tady bylo chybějící .unwrap()

    let y = iris.target.clone();

    // Train/test split
    let (x_train, x_test, y_train, y_test) =
        train_test_split(&x, &y, 0.2, true, Some(42));

    // Random Forest
    let model = RandomForestClassifier::fit(
        &x_train,
        &y_train,
        Default::default(),
    ).unwrap();

    let y_pred = model.predict(&x_test).unwrap();
    let acc = accuracy(&y_test, &y_pred);

    println!("Accuracy: {:.2}%", acc * 100.0);
}
''', deps='''
smartcore = { version = "0.5.5", features = ["datasets"] }
''')

Accuracy: 96.67%

